# ДЗ 12. Интегрирование рациональных функций по методу Остроградского

## Теоретические задания

### 1) Сформулируйте задачу Остроградского и опишите по шагам метод её решения.

**Задача Остроградского.** Задана рациональная функция с целыми коэффициентами. Требуется найти её интеграл в виде элементарного выражения, содержащего в качестве чисел только рациональные и вещественные алгебраические числа.

**Метод решения (3 шага):**

1. **Приведение к правильной дроби.** Делим числитель на знаменатель: $f = u + g/h$, где $\deg g < \deg h$. Многочлен $u$ интегрируется тривиально.

2. **Декомпозиция Остроградского.** Раскладываем знаменатель $h$ на неприводимые множители в $\mathbb{Z}[x]$: $h = p_1^{n_1} \cdots p_r^{n_r}$. Тогда
$$\frac{g}{h} = \frac{q_l}{p_1 \cdots p_r} + D\!\left(\frac{q_a}{p_1^{n_1-1} \cdots p_r^{n_r-1}}\right),$$
   где $q_l, q_a \in \mathbb{Q}[x]$ находятся методом неопределённых коэффициентов (СЛАУ с рациональными коэффициентами). Первое слагаемое — логарифмическая часть, второе — алгебраическая часть (рациональная функция, входит в ответ как есть).

3. **Интегрирование логарифмической части.** Знаменатель $p_1 \cdots p_r$ не имеет кратных корней в $\mathbb{C}$, поэтому раскладываем на простейшие дроби в поле частных кольца $\mathbb{A}[x]$ и интегрируем каждую через `pfdintegral`.

### 2) Всегда ли интеграл от рациональной функции с целыми коэффициентами является элементарной функцией?

Да, всегда. По теореме 27 (Остроградского) интеграл от рациональной функции с целыми коэффициентами представляет собой сумму рациональной функции с рациональными коэффициентами и линейной комбинации логарифмов и арктангенсов от рациональных функций с коэффициентами из $\mathbb{A}$. Всё это — элементарные функции.

### 3) Какому полю принадлежат коэффициенты трансцендентных функций, входящих в первообразную рациональной функции с целыми коэффициентами?

Коэффициенты трансцендентных функций (логарифмов и арктангенсов) принадлежат полю вещественных алгебраических чисел $\mathbb{A}$. При этом рациональная (алгебраическая) часть интеграла и числители/знаменатели в декомпозиции Остроградского принадлежат $\mathbb{Q}$.

---
## Практические задания

In [ ]:
var('x')

Вспомогательные функции из лекций (`pfdintegral`, `ostrogradski`):

In [ ]:
def pfdintegral(f, x):
    g = (f).numerator()
    h = (f).denominator()
    if AA[x](h).degree() == 1:
        return g*ln(abs(x + h.subs(x=0)))
    else:
        a0 = h.subs(x=0)
        a1 = diff(h, x).subs(x=0)
        disc = a1^2 - 4*a0
        if disc == 0:
            return -g/(x + a1/2)
        b0 = g.subs(x=0)
        b1 = diff(g, x)
        s = sqrt(-disc)
        return 1/2*b1*log(x^2 + a1*x + a0) - (a1*b1 - 2*b0)*arctan((2*x + a1)/s)/s

In [ ]:
def ostrogradski(f, x):
    K = FractionField(ZZ[x])
    g = K(f).numerator()
    h = K(f).denominator()
    L = list(ZZ[x](h).factor())
    hh = prod([p^(m-1) for (p, m) in L])
    n = hh.degree()
    if n == 0:
        return [0, f]
    A = var(['A' + str(i) for i in range(n)])
    if n == 1: A = [A]
    gg = sum([A[i]*x^i for i in range(n)])
    h3 = ZZ[x](prod([p for (p, m) in L]))
    m = h3.degree()
    B = var(['B' + str(i) for i in range(m)])
    if m == 1: B = [B]
    g3 = sum([B[i]*x^i for i in range(m)])
    F = (diff(SR(gg/hh), x) + SR(g3/h3) - f).numerator().expand()
    deg = ZZ[x](h).degree()
    eqs = [F.coefficient(x, k) for k in range(deg)]
    unknowns = list(A) + list(B)
    S = solve(eqs, unknowns)[0]
    return [(gg).subs(S)/hh, (g3).subs(S)/h3]

### Задача 1. Реализация решения задачи Остроградского.

На входе — рациональная функция с целыми коэффициентами, на выходе — её первообразная.

In [ ]:
def ostrogradski_integral(f, x):
    """Интеграл рациональной функции с целыми коэфф. по методу Остроградского."""
    g = SR(f).numerator()
    h = SR(f).denominator()
    # Шаг 1: выделяем полиномиальную часть
    u, r = QQ[x](g).quo_rem(QQ[x](h))
    # Интеграл полиномиальной части
    F = sum(u[k]*x^(k+1)/(k+1) for k in range(u.degree()+1)) if u.degree() >= 0 else SR(0)
    if r == 0:
        return F
    # Шаг 2: декомпозиция Остроградского
    [alg, logpart] = ostrogradski(r/h, x)
    F += alg
    # Шаг 3: интегрирование логарифмической части
    if logpart != 0:
        PFD = FractionField(AA[x])(logpart).partial_fraction_decomposition()
        F += sum(pfdintegral(fr, x) for fr in PFD[1])
    return F

In [ ]:
# Тест: пример 44 из лекции
f = x^10/(x^3 + 8)^3
F = ostrogradski_integral(f, x)
print(F)
print('Проверка:', (F.diff(x) - f).subs(x=1.0).n())

### Задача 2. Найти первообразную $\displaystyle\int \frac{dx}{(x^3+1)^2(x^5+2)^3}$, без комплексных чисел.

In [ ]:
f = 1/((x^3 + 1)^2*(x^5 + 2)^3)
F = ostrogradski_integral(f, x)
print(F)

In [ ]:
# Проверка
print('Проверка:', (F.diff(x) - f).subs(x=0.5).n())

### Задача 3. График первообразной $\displaystyle\int \frac{x^9\,dx}{(x^4+x+1)^2}$ на $-10 < x < 10$.

In [ ]:
f = x^9/(x^4 + x + 1)^2
F = ostrogradski_integral(f, x)
print('Первообразная:', F)

In [ ]:
plot(F, (x, -10, 10), ymin=-15, ymax=15)

### Задача 4. Вычислить $\displaystyle\int_1^2 \frac{x^6(x+1)^5(x+2)^4\,dx}{(x^4-2)^2(x-3)^3}$ с точностью 100 знаков.

In [ ]:
f = x^6*(x+1)^5*(x+2)^4/((x^4-2)^2*(x-3)^3)
F = ostrogradski_integral(f, x)
val = F.subs(x=2) - F.subs(x=1)
print(val.n(digits=100))

In [ ]:
# Сравнение с numerical_integral
numerical_integral(f, (1, 2))

### Задача 5. Модифицировать реализацию метода Остроградского с `radical_expression`.

In [ ]:
def pfdintegral_rad(f, x):
    g = (f).numerator()
    h = (f).denominator()
    if AA[x](h).degree() == 1:
        c = AA(g).radical_expression()
        a = AA(h.subs(x=0)).radical_expression()
        return c*ln(abs(x + a))
    else:
        a0 = AA(h.subs(x=0)).radical_expression()
        a1 = AA(diff(h, x).subs(x=0)).radical_expression()
        disc = a1^2 - 4*a0
        if disc == 0:
            c = AA(g).radical_expression()
            return -c/(x + a1/2)
        b0 = AA(g.subs(x=0)).radical_expression()
        b1 = AA(diff(g, x)).radical_expression()
        s = sqrt(-disc)
        return 1/2*b1*log(x^2 + a1*x + a0) - (a1*b1 - 2*b0)*arctan((2*x + a1)/s)/s

def ostrogradski_integral_rad(f, x):
    """Метод Остроградского с коэффициентами в радикалах."""
    g = SR(f).numerator()
    h = SR(f).denominator()
    u, r = QQ[x](g).quo_rem(QQ[x](h))
    F = sum(u[k]*x^(k+1)/(k+1) for k in range(u.degree()+1)) if u.degree() >= 0 else SR(0)
    if r == 0:
        return F
    [alg, logpart] = ostrogradski(r/h, x)
    F += alg
    if logpart != 0:
        PFD = FractionField(AA[x])(logpart).partial_fraction_decomposition()
        F += sum(pfdintegral_rad(fr, x) for fr in PFD[1])
    return F

In [ ]:
# Тест
f = x/(x^3 + 8)^3
F = ostrogradski_integral_rad(f, x)
print(F)

### Задача 6. Выразить $\displaystyle\int_0^1 \frac{dx}{(x^4-2)^3}$ как символьное выражение с радикалами.

In [ ]:
f = 1/(x^4 - 2)^3
F = ostrogradski_integral_rad(f, x)
val = F.subs(x=1) - F.subs(x=0)
print('Символьное выражение:')
print(val.simplify())
print()
print('Числовое значение:', val.n(digits=50))
print('numerical_integral:', numerical_integral(f, (0, 1)))

### Задача 7. Выразить $\displaystyle\int_0^1 \frac{x\,dx}{(x^3+x+8)^3}$ как символьное выражение с радикалами.

In [ ]:
f = x/(x^3 + x + 8)^3
F = ostrogradski_integral_rad(f, x)
val = F.subs(x=1) - F.subs(x=0)
print('Символьное выражение:')
print(val.simplify())
print()
print('Числовое значение:', val.n(digits=50))
print('numerical_integral:', numerical_integral(f, (0, 1)))